# Fare Prediction

In [3]:
import numpy as np
import pandas as pd 
import math
import joblib   
from xgboost import XGBRegressor
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score , mean_absolute_percentage_error
from sklearn.ensemble import RandomForestClassifier


In [4]:
PROCESSED_DIR = Path("../data/processed")
df = pd.read_parquet(PROCESSED_DIR / "prediction_anomaly_data.parquet")
df.head()

,tpep_pickup_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,...,payment_1,payment_2,payment_3,payment_4,ratecode_1.0,ratecode_2.0,ratecode_3.0,ratecode_4.0,ratecode_5.0,ratecode_6.0
0,2024-12-31 23:48:29,1.0,2.53,1.0,186,79,2,16.3,1.0,0.5,...,False,True,False,False,True,False,False,False,False,False
1,2024-12-31 23:52:40,1.0,6.72,1.0,114,151,1,34.5,1.0,0.5,...,True,False,False,False,True,False,False,False,False,False
2,2025-01-01 00:00:00,1.0,6.40,1.0,211,164,1,52.0,3.5,0.5,...,True,False,False,False,True,False,False,False,False,False
3,2025-01-01 00:00:19,1.0,10.08,1.0,132,61,1,44.3,1.0,0.5,...,True,False,False,False,True,False,False,False,False,False
4,2025-01-01 00:01:06,1.0,0.70,1.0,162,162,1,9.3,3.5,0.5,...,True,False,False,False,True,False,False,False,False,False


In [5]:
fare_features = [
    'passenger_count', 'trip_distance',
    'PULocationID', 'DOLocationID',
    'pickup_hour', 'pickup_dayofweek', 'pickup_month',
    'is_weekend', 'is_rush_hour', 'is_night',
    'congestion_surcharge', 'cbd_congestion_fee', 'Airport_fee',
    'payment_1', 'payment_2', 'payment_3', 'payment_4',
    'ratecode_1.0', 'ratecode_2.0', 'ratecode_3.0',
    'ratecode_4.0', 'ratecode_5.0', 'ratecode_6.0', 'fare_amount'
]


df = df[fare_features]

In [6]:
x = df.drop("fare_amount", axis=1)
y = df["fare_amount"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [8]:
xgb  = XGBRegressor(n_estimators=100, random_state=42, max_depth=7, learning_rate=0.1 , verbose=1)

In [9]:
xgb.fit(X_train, y_train)

C:\Users\Click\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:200: UserWarning: [18:53:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [10]:
y_pred = xgb.predict(X_test)

In [ ]:
# Saving model 
joblib.dump(xgb, "../Models/xgboost_fare_model.pkl")
print(" XGBoost fare model saved")

 XGBoost fare model saved


In [15]:
# Calculate errors

mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

# Baseline: always predict the average fare
avg_fare = y_train.mean()
baseline_mae = mean_absolute_error(y_test, [avg_fare] * len(y_test))

print("\n" + "-"*60)
print("  FARE PREDICTION MODEL – PERFORMANCE REPORT")
print("-"*60)

print("\n MEAN ABSOLUTE ERROR (MAE): ${:.2f}".format(mae))
print("   → On average, our prediction is off by ${:.2f}.".format(mae))
print("   → Example: If the actual fare is $15, we might predict between ${:.2f} and ${:.2f}.".format(15-mae, 15+mae))

print("\n ROOT MEAN SQUARED ERROR (RMSE): ${:.2f}".format(rmse))
print("   → This is similar to MAE but penalizes large mistakes more heavily.")
print("   → A small gap between MAE and RMSE means our errors are consistent (no huge outliers).")

print("\n R² SCORE: {:.2%}".format(r2))
print("   → Our model explains {:.0%} of the variation in taxi fares.".format(r2))
print("   → The remaining {:.0%} is due to factors we don't know (traffic, driver, weather, etc.).".format(1-r2))

print("\n MEAN ABSOLUTE PERCENTAGE ERROR (MAPE): {:.1f}%".format(mape))
print("   → Typically, our prediction is off by about {:.1f}% of the actual fare.".format(mape))
print("   → Example: For a $20 fare, we might be off by ${:.2f} on average.".format(20 * mape/100))

print("\n COMPARED TO A DUMMY MODEL (always guesses ${:.2f})".format(avg_fare))
print("   → Our MAE is ${:.2f} vs. dummy MAE ${:.2f}".format(mae, baseline_mae))
print("   → That's a {:.0f}% improvement – much better than just guessing the average.".format((1 - mae/baseline_mae)*100))

print("\n VERDICT: The model is accurate enough for real‑world use (e.g., fare estimation apps).")
print("-"*60)


------------------------------------------------------------
  FARE PREDICTION MODEL – PERFORMANCE REPORT
------------------------------------------------------------

 MEAN ABSOLUTE ERROR (MAE): $2.05
   → On average, our prediction is off by $2.05.
   → Example: If the actual fare is $15, we might predict between $12.95 and $17.05.

 ROOT MEAN SQUARED ERROR (RMSE): $4.47
   → This is similar to MAE but penalizes large mistakes more heavily.
   → A small gap between MAE and RMSE means our errors are consistent (no huge outliers).

 R² SCORE: 94.04%
   → Our model explains 94% of the variation in taxi fares.
   → The remaining 6% is due to factors we don't know (traffic, driver, weather, etc.).

 MEAN ABSOLUTE PERCENTAGE ERROR (MAPE): 32.4%
   → Typically, our prediction is off by about 32.4% of the actual fare.
   → Example: For a $20 fare, we might be off by $6.49 on average.

 COMPARED TO A DUMMY MODEL (always guesses $19.72)
   → Our MAE is $2.05 vs. dummy MAE $12.04
   → That's a